In [20]:
# ==============================================================================
#  Project 06: Fabric Defect Detection for Quality Control
#  Applied Machine Learning — EDGE Series, DUET Gazipur
#  Single-file Kaggle Notebook — AUTO-DETECT DATASET VERSION
# ==============================================================================

import subprocess
subprocess.run(["pip", "install", "gradio", "opencv-python-headless", "--quiet"], check=True)

import os, shutil, random, warnings
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image, ImageFilter
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import gradio as gr

warnings.filterwarnings("ignore")
tf.random.set_seed(42)
np.random.seed(42)
random.seed(42)

print("TensorFlow :", tf.__version__)
print("Gradio     :", gr.__version__)
print("GPU        :", tf.config.list_physical_devices("GPU"))

# ==============================================================================
#  STEP 0 — AUTO-DETECT DATASET FOLDER STRUCTURE
# ==============================================================================

# Search the entire /kaggle/input directory
DATASET_ROOT = "/kaggle/input"
image_exts = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

print("\n" + "="*60)
print("  SCANNING KAGGLE INPUT FOR IMAGES")
print("="*60)

defect_folders = {}
for dirpath, dirnames, filenames in os.walk(DATASET_ROOT):
    img_files = [f for f in filenames if os.path.splitext(f)[1].lower() in image_exts]
    if len(img_files) > 10:  # Only count folders with a reasonable number of images
        folder_name = os.path.basename(dirpath).lower()
        defect_folders[folder_name] = dirpath
        print(f"  Found: {os.path.relpath(dirpath, DATASET_ROOT)}/  ({len(img_files)} images)")

if not defect_folders:
    raise FileNotFoundError(
        "No image folders found in /kaggle/input! "
        "Please ensure you added the DATASET, not a Notebook."
    )

# ==============================================================================
#  SECTION 1 — DATASET PREPARATION & EDA
# ==============================================================================

SPLIT_ROOT  = "/kaggle/working/data_split"
MODEL_PATH  = "/kaggle/working/best_model.keras"
EXAMPLE_DIR = "/kaggle/working/examples"
IMG_SIZE    = 224
BATCH_SIZE  = 32
EPOCHS      = 25

def gather_all_pairs(folders_dict):
    pairs = []
    for fname, fpath in folders_dict.items():
        for f in os.listdir(fpath):
            if os.path.splitext(f)[1].lower() in image_exts:
                pairs.append((os.path.join(fpath, f), 1))  # 1 = defective
    
    synth_dir = "/kaggle/working/synth_non_defective"
    os.makedirs(synth_dir, exist_ok=True)
    
    n_synth = len(pairs)
    source_paths = [p for p, _ in pairs]
    chosen = random.choices(source_paths, k=n_synth)
    
    print(f"\n  Synthesising {n_synth} non-defective images (blurring defects)...")
    for i, src in enumerate(chosen):
        try:
            img = Image.open(src).convert("RGB")
            w, h = img.size
            margin = max(w // 6, 10)
            crop = img.crop((margin, margin, w - margin, h - margin)).resize((w, h), Image.BILINEAR)
            blurred = crop.filter(ImageFilter.GaussianBlur(radius=random.uniform(4, 7)))
            
            arr = np.array(blurred, dtype=np.float32)
            arr = np.clip(arr * random.uniform(0.9, 1.1) + random.uniform(-10, 10), 0, 255).astype(np.uint8)
            
            save_path = os.path.join(synth_dir, f"nondef_{i:06d}.png")
            Image.fromarray(arr).save(save_path)
            pairs.append((save_path, 0))
        except Exception:
            pass
            
    random.shuffle(pairs)
    return pairs

pairs = gather_all_pairs(defect_folders)

def split_and_copy(pairs):
    paths  = [p[0] for p in pairs]
    labels = [p[1] for p in pairs]

    X_tv, X_test, y_tv, y_test = train_test_split(paths, labels, test_size=0.15, stratify=labels, random_state=42)
    X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, test_size=0.176, stratify=y_tv, random_state=42)

    if os.path.exists(SPLIT_ROOT): shutil.rmtree(SPLIT_ROOT)
    
    for split, (xs, ys) in [("train", (X_train, y_train)), ("val", (X_val, y_val)), ("test", (X_test, y_test))]:
        for x, y in zip(xs, ys):
            dest = os.path.join(SPLIT_ROOT, split, "defective" if y==1 else "non_defective")
            os.makedirs(dest, exist_ok=True)
            shutil.copy2(x, os.path.join(dest, f"{random.randint(1000,9999)}_{os.path.basename(x)}"))
            
    print(f"\n  Split: Train={len(X_train)} | Val={len(X_val)} | Test={len(X_test)}")
    return X_train, X_val, X_test

split_and_copy(pairs)

# ==============================================================================
#  SECTION 2 — MODEL & TRAINING (FAST VERSION)
# ==============================================================================

def preprocess(image, label):
    return tf.cast(tf.image.resize(image, [IMG_SIZE, IMG_SIZE]), tf.float32) / 255.0, label

train_ds = keras.utils.image_dataset_from_directory(
    os.path.join(SPLIT_ROOT, "train"), label_mode="binary", image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE
).map(preprocess).cache().prefetch(tf.data.AUTOTUNE)

val_ds = keras.utils.image_dataset_from_directory(
    os.path.join(SPLIT_ROOT, "val"), label_mode="binary", image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE
).map(preprocess).cache().prefetch(tf.data.AUTOTUNE)

test_ds = keras.utils.image_dataset_from_directory(
    os.path.join(SPLIT_ROOT, "test"), label_mode="binary", image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE
).map(preprocess).cache().prefetch(tf.data.AUTOTUNE)

inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = layers.Conv2D(32, 3, padding="same", activation="relu")(inputs)
x = layers.MaxPooling2D(2)(x)
x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
x = layers.MaxPooling2D(2)(x)
x = layers.Conv2D(128, 3, padding="same", activation="relu", name="last_conv")(x) # GRAD-CAM LAYER
x = layers.MaxPooling2D(2)(x)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = keras.Model(inputs, outputs)
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy", keras.metrics.AUC(name="auc")])

print("\nStarting Training...")
model.fit(
    train_ds, validation_data=val_ds, epochs=15, 
    callbacks=[keras.callbacks.ModelCheckpoint(MODEL_PATH, save_best_only=True, monitor="val_auc")]
)
best_model = keras.models.load_model(MODEL_PATH)

# ==============================================================================
#  SECTION 3 — GRAD-CAM & GUI
# ==============================================================================

def compute_gradcam(mdl, img_array):
    grad_model = tf.keras.Model([mdl.inputs], [mdl.get_layer("last_conv").output, mdl.output])
    img_tensor = tf.cast(img_array, tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(img_tensor)
        conv_outputs, preds = grad_model(img_tensor)
        loss = preds[:, 0]
    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = tf.reduce_sum(tf.multiply(pooled_grads, conv_outputs[0]), axis=-1)
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

def predict(image):
    if image is None: return None, None, "No image", ""
    pil_image = Image.fromarray(image.astype(np.uint8)) if isinstance(image, np.ndarray) else image
    img_r = pil_image.convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    img_arr = np.expand_dims(np.array(img_r, dtype=np.float32) / 255.0, axis=0)
    
    prob = float(best_model.predict(img_arr, verbose=0)[0][0])
    label = "Defective" if prob > 0.5 else "Non-Defective"
    
    hm = compute_gradcam(best_model, img_arr)
    hm_colored = cv2.applyColorMap(np.uint8(255 * cv2.resize(hm, pil_image.size)), cv2.COLORMAP_JET)
    overlay = Image.fromarray(np.clip((0.6 * np.array(pil_image) + 0.4 * cv2.cvtColor(hm_colored, cv2.COLOR_BGR2RGB)), 0, 255).astype(np.uint8))
    
    badge = f"<h2 style='color:{'red' if prob>0.5 else 'green'};text-align:center;'>{label} ({prob:.1%})</h2>"
    return pil_image, overlay, badge, "Analysis complete."

# Prepare examples
os.makedirs(EXAMPLE_DIR, exist_ok=True)
for f in os.listdir(os.path.join(SPLIT_ROOT, "test", "defective"))[:3]:
    shutil.copy(os.path.join(SPLIT_ROOT, "test", "defective", f), EXAMPLE_DIR)

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# Fabric Defect Detection | Project 06")
    with gr.Row():
        inp = gr.Image(type="pil", label="Upload Patch")
        with gr.Column():
            res = gr.HTML()
            txt = gr.Textbox(label="Status")
    with gr.Row():
        orig = gr.Image(label="Original")
        over = gr.Image(label="Grad-CAM Heatmap")
    
    ex = gr.Examples([os.path.join(EXAMPLE_DIR, f) for f in os.listdir(EXAMPLE_DIR)], inp)
    btn = gr.Button("Analyse", variant="primary")
    btn.click(predict, inputs=inp, outputs=[orig, over, res, txt])

demo.launch(server_name="0.0.0.0", server_port=7860, share=True)

TensorFlow : 2.19.0
Gradio     : 5.50.0
GPU        : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]

  SCANNING KAGGLE INPUT FOR IMAGES
  Found: datasets/kaustubhdikshit/neu-surface-defect-database/NEU-DET/validation/images/inclusion/  (60 images)
  Found: datasets/kaustubhdikshit/neu-surface-defect-database/NEU-DET/validation/images/scratches/  (60 images)
  Found: datasets/kaustubhdikshit/neu-surface-defect-database/NEU-DET/validation/images/pitted_surface/  (60 images)
  Found: datasets/kaustubhdikshit/neu-surface-defect-database/NEU-DET/validation/images/patches/  (60 images)
  Found: datasets/kaustubhdikshit/neu-surface-defect-database/NEU-DET/validation/images/crazing/  (60 images)
  Found: datasets/kaustubhdikshit/neu-surface-defect-database/NEU-DET/validation/images/rolled-in_scale/  (60 images)
  Found: datasets/kaustubhdikshit/neu-surface-defect-database/NEU-DET/train/images/inclusion/  (240

I0000 00:00:1778828735.011736      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1778828735.017993      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Found 431 files belonging to 2 classes.
Found 432 files belonging to 2 classes.

Starting Training...
Epoch 1/15


I0000 00:00:1778828739.771524     185 service.cc:152] XLA service 0x7bc5383106a0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778828739.771582     185 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1778828739.771589     185 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1778828740.305412     185 cuda_dnn.cc:529] Loaded cuDNN version 91002


 2/64 ━━━━━━━━━━━━━━━━━━━━ 9s 145ms/step - accuracy: 0.4844 - auc: 0.3503 - loss: 0.7137

I0000 00:00:1778828745.848036     185 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


63/64 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - accuracy: 0.5140 - auc: 0.5540 - loss: 0.6902

2026-05-15 07:05:50.355348: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-15 07:05:50.491594: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


64/64 ━━━━━━━━━━━━━━━━━━━━ 17s 153ms/step - accuracy: 0.5154 - auc: 0.5565 - loss: 0.6897 - val_accuracy: 0.7958 - val_auc: 0.9998 - val_loss: 0.5811
Epoch 2/15
64/64 ━━━━━━━━━━━━━━━━━━━━ 3s 54ms/step - accuracy: 0.7405 - auc: 0.8484 - loss: 0.5421 - val_accuracy: 0.9745 - val_auc: 1.0000 - val_loss: 0.2477
Epoch 3/15
64/64 ━━━━━━━━━━━━━━━━━━━━ 4s 55ms/step - accuracy: 0.8794 - auc: 0.9454 - loss: 0.3175 - val_accuracy: 0.9930 - val_auc: 0.9942 - val_loss: 0.2299
Epoch 4/15
64/64 ━━━━━━━━━━━━━━━━━━━━ 3s 54ms/step - accuracy: 0.9108 - auc: 0.9655 - loss: 0.2556 - val_accuracy: 0.9930 - val_auc: 0.9987 - val_loss: 0.2001
Epoch 5/15
64/64 ━━━━━━━━━━━━━━━━━━━━ 3s 54ms/step - accuracy: 0.8327 - auc: 0.8962 - loss: 0.3811 - val_accuracy: 0.9861 - val_auc: 0.9987 - val_loss: 0.1220
Epoch 6/15
64/64 ━━━━━━━━━━━━━━━━━━━━ 3s 55ms/step - accuracy: 0.8912 - auc: 0.9548 - loss: 0.2554 - val_accuracy: 0.9420 - val_auc: 0.9969 - val_loss: 0.1487
Epoch 7/15
64/64 ━━━━━━━━━━━━━━━━━━━━ 4s 55ms/step - ac